# MOD-07 · NB-08 — Capstone: Spatial Health Inequity Atlas
### Health Analytics with Python · Module 07: Geospatial & Spatial Epidemiology
---
**This capstone integrates NB-01 through NB-07 into a comprehensive spatial health inequity analysis.**

**Research Question:** Where are counties with high CVD mortality, high pollution burden, AND poor healthcare access simultaneously concentrated — and how does this geographic intersection relate to poverty?

**Pipeline stages:**
1. Data assembly: choropleth mapping of all key variables
2. Spatial autocorrelation: Moran's I + LISA for CVD mortality and poverty
3. Spatial cluster detection: scan statistic for CVD hot spots
4. Spatial regression: OLS → spatial lag model comparing estimates
5. Empirical Bayes disease mapping: smoothed SMR surface
6. Environmental justice: pollution × race/income analysis
7. Access mapping: 2SFCA + HPSA identification
8. Compound disadvantage index: counties with multiple burdens
9. Publication-quality 8-panel figure (300 dpi) + methods narrative

**Estimated time:** 4 hours | **Level:** Advanced


## 0. Setup & Synthetic County Dataset

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec, seaborn as sns
from scipy.ndimage import gaussian_filter
from scipy.spatial import cKDTree
from scipy.spatial.distance import cdist
from scipy import stats
import statsmodels.api as sm
import warnings; warnings.filterwarnings("ignore")
import os; os.makedirs("/tmp/mod07", exist_ok=True)
plt.rcParams.update({"figure.dpi":120,"savefig.dpi":300,"figure.facecolor":"white",
                     "axes.spines.top":False,"axes.spines.right":False})
SEED=42; np.random.seed(SEED)
NROW,NCOL=10,20; N=NROW*NCOL
row_idx=np.repeat(np.arange(NROW),NCOL); col_idx=np.tile(np.arange(NCOL),NROW)
lon=-120+col_idx*2.5+np.random.normal(0,0.3,N)
lat=28+row_idx*2.0+np.random.normal(0,0.2,N)
pct_poverty    = 10+8*np.random.beta(2,5,N)+3*np.sin(lon/20)
pct_uninsured  = 8+6*np.random.beta(2,4,N)+0.3*pct_poverty
pct_minority   = (20+15*np.random.beta(2,3,N)).clip(0,100)
pct_rural      = (30+40*np.random.beta(3,3,N)).clip(0,100)
population     = np.random.lognormal(10.5,1.0,N).astype(int)
pm25_base      = 8+4*(lon-lon.min())/(lon.max()-lon.min())
def sp(v,s=2):
    g=np.zeros((NROW,NCOL))
    for r,c,val in zip(row_idx,col_idx,v): g[r,c]=val
    return gaussian_filter(g,sigma=s)[row_idx,col_idx]
pm25 = (pm25_base+sp(np.random.gamma(1,1,N),sigma=1.5)+np.random.normal(0,1,N)).clip(3,28)
spatial_risk = sp(np.random.normal(0,1,N),sigma=2.5)
# Plant two true hot spots
cluster1 = (lon>-105)&(lon<-95)&(lat>30)&(lat<36)
cluster2 = (lon>-85)&(lon<-75)&(lat>38)&(lat<44)
base_cvd = (180+1.2*pct_poverty+0.8*pct_uninsured+0.5*pm25+12*spatial_risk+np.random.normal(0,10,N))
cvd_rate = base_cvd.copy()
cvd_rate[cluster1] += 35; cvd_rate[cluster2] += 25
expected=(cvd_rate/100000)*population*5
observed=np.random.poisson(expected).astype(int); smr=observed/expected.clip(1)
# Hospitals
np.random.seed(5); N_H=40
hlon=np.random.uniform(lon.min(),lon.max(),N_H)
hlat=np.random.uniform(lat.min(),lat.max(),N_H)
hbeds=np.random.randint(50,600,N_H)
df=pd.DataFrame({"county_id":[f"C{i:03d}" for i in range(N)],
    "lon":lon,"lat":lat,"row":row_idx,"col":col_idx,
    "pct_poverty":pct_poverty,"pct_uninsured":pct_uninsured,
    "pct_minority":pct_minority,"pct_rural":pct_rural,
    "pm25":pm25,"population":population,"cvd_rate":cvd_rate,
    "observed":observed,"expected":expected,"smr":smr,
    "cluster1":cluster1,"cluster2":cluster2})
print(f"N={N} counties | CVD={cvd_rate.mean():.1f}/100k | PMt2.5={pm25.mean():.1f} µg/m³")


## 1. Build Spatial Weights

In [ ]:
def build_queen(row_idx,col_idx):
    N=len(row_idx); W=np.zeros((N,N))
    for i in range(N):
        for j in range(N):
            if i!=j and max(abs(row_idx[i]-row_idx[j]),abs(col_idx[i]-col_idx[j]))<=1:
                W[i,j]=1
    rs=W.sum(axis=1,keepdims=True); rs[rs==0]=1; return W/rs, (W>0).astype(float)

W, W_bin = build_queen(df.row.values, df.col.values)
print(f"Spatial weights: N={N}x{N} | Mean neighbours: {(W>0).sum(axis=1).mean():.1f}")


## 2. Global & Local Moran's I

In [ ]:
def morans_i(y,W,n_perm=499,seed=42):
    n=len(y); z=y-y.mean(); S0=W.sum()
    I=(n/S0)*(z@(W@z))/(z@z)
    rng=np.random.default_rng(seed)
    Ip=[(n/S0)*(rng.permutation(z)@(W@rng.permutation(z)))/(z@z) for _ in range(n_perm)]
    p=(np.abs(Ip)>=abs(I)).mean()
    return I,p

def local_morans(y,W,n_perm=499,seed=42):
    n=len(y); z=y-y.mean(); vz=z.var()
    Wz=W@z; Il=z*Wz/vz
    rng=np.random.default_rng(seed)
    Ip=np.array([(rng.permutation(z)*(W@rng.permutation(z))/vz) for _ in range(n_perm)])
    pv=(np.abs(Ip)>=np.abs(Il)).mean(axis=0)
    zstd=z/z.std(); zlag=Wz/z.std()
    quad=np.where((zstd>0)&(zlag>0),"HH",np.where((zstd<0)&(zlag<0),"LL","Other"))
    return np.where(pv<0.05,quad,"NS"), Il, pv

I_cvd,p_cvd   = morans_i(df.cvd_rate.values,W)
I_pov,p_pov   = morans_i(df.pct_poverty.values,W)
I_pm25,p_pm25 = morans_i(df.pm25.values,W)
print(f"Global Moran's I:")
print(f"  CVD rate : {I_cvd:.4f} (p={p_cvd:.4f})")
print(f"  Poverty  : {I_pov:.4f} (p={p_pov:.4f})")
print(f"  PM2.5    : {I_pm25:.4f} (p={p_pm25:.4f})")

lisa_cvd, _, p_lisa = local_morans(df.cvd_rate.values,W)
df["lisa_cvd"]  = lisa_cvd
df["lisa_p_cvd"]= p_lisa
print(f"\nLISA CVD: HH={sum(lisa_cvd=='HH')} LL={sum(lisa_cvd=='LL')} NS={sum(lisa_cvd=='NS')}")


## 3. Spatial Cluster Detection (Scan Statistic)

In [ ]:
def scan_stat(df_in, max_frac=0.4, n_perm=99, seed=42):
    coords=np.column_stack([df_in.lon,df_in.lat])
    O_tot=df_in.observed.sum(); E_tot=df_in.expected.sum()
    n=len(df_in); max_pop=df_in.population.sum()*max_frac
    tree=cKDTree(coords)
    def llr(Oz,Ez):
        if Oz<=0 or Ez<=0 or Oz<=Ez: return 0
        Oo=O_tot-Oz; Eo=E_tot-Ez
        if Oo<=0 or Eo<=0 or Oo<=Eo: return 0
        return max(0,Oz*np.log(Oz/Ez)+Oo*np.log(Oo/Eo))
    best_llr=0; best_c=0; best_r=0; best_in=[]
    for ci in range(n):
        _, sidx=tree.query(coords[ci],k=n)
        Oz=0; Ez=0; popz=0
        for k in range(1,n):
            j=sidx[k]; popz+=df_in.population.iloc[j]
            if popz>max_pop: break
            Oz+=df_in.observed.iloc[j]; Ez+=df_in.expected.iloc[j]
            l=llr(Oz,Ez)
            if l>best_llr:
                best_llr=l; best_c=ci; best_r=np.linalg.norm(coords[ci]-coords[j]); best_in=sidx[1:k+1].tolist()
    rng=np.random.default_rng(seed)
    mc=[]
    E_arr=df_in.expected.values
    for _ in range(n_perm):
        Op=rng.multinomial(O_tot,E_arr/E_tot); bl=0
        for ci in range(n):
            _,sidx=tree.query(coords[ci],k=n)
            Oz=0; Ez=0; popz=0
            for k in range(1,n):
                j=sidx[k]; popz+=df_in.population.iloc[j]
                if popz>max_pop: break
                Oz+=Op[j]; Ez+=E_arr[j]; l=llr(Oz,Ez)
                if l>bl: bl=l
        mc.append(bl)
    return best_llr, best_c, best_r, best_in, (np.array(mc)>=best_llr).mean()

print("Running scan statistic...")
best_llr,best_c,best_r,best_in,p_scan = scan_stat(df)
df["in_scan_cluster"] = [i in set(best_in) for i in range(N)]
print(f"Primary cluster: LLR={best_llr:.2f} p={p_scan:.3f} n={len(best_in)} counties")


## 4. Spatial Regression

In [ ]:
# OLS
X=sm.add_constant(df[["pct_poverty","pct_uninsured","pm25"]])
ols=sm.OLS(df.cvd_rate,X).fit()
ols_pov=ols.params["pct_poverty"]
I_resid,p_resid=morans_i(ols.resid.values,W,n_perm=299)
print(f"OLS: poverty β={ols_pov:.3f} | Moran I(resid)={I_resid:.4f} (p={p_resid:.4f})")

# Spatial Lag Model (simplified via grid search)
y_arr=df.cvd_rate.values; X_arr=X.values; n=N; I_n=np.eye(n)
rho_grid=np.linspace(-0.95,0.95,60); ll_grid=[]
for rho in rho_grid:
    A=I_n-rho*W; Ay=A@y_arr
    b=np.linalg.lstsq(X_arr,Ay,rcond=None)[0]; e=Ay-X_arr@b; s2=(e@e)/n
    sign,ld=np.linalg.slogdet(A)
    if sign<=0: ll_grid.append(-1e10); continue
    ll_grid.append(-n/2*np.log(2*np.pi)-n/2*np.log(s2)+ld-n/2)
rho_best=rho_grid[np.argmax(ll_grid)]
A=I_n-rho_best*W; Ay=A@y_arr
b_slm=np.linalg.lstsq(X_arr,Ay,rcond=None)[0]
slm_pov=b_slm[X.columns.get_loc("pct_poverty")]
print(f"SLM: rho={rho_best:.3f} | poverty β={slm_pov:.3f} | bias vs OLS: {slm_pov-ols_pov:+.3f}")
print(f"True poverty coefficient: 1.20")


## 5. Empirical Bayes Disease Mapping

In [ ]:
# Gamma-Poisson EB smoothing
smr_mean=df.smr.mean(); smr_var=df.smr.var()
extra_var=max(smr_var-smr_mean/df.expected.mean(),0.01)
alpha_hat=smr_mean**2/extra_var; beta_hat=smr_mean/extra_var
df["smr_eb"]=(df.observed+alpha_hat)/(df.expected+beta_hat)
print(f"EB smoothing: raw SMR SD={df.smr.std():.4f} -> smoothed SD={df.smr_eb.std():.4f}")
print(f"(variance reduction: {(1-df.smr_eb.std()/df.smr.std())*100:.1f}%)")


## 6. Access Metrics & Compound Disadvantage Index

In [ ]:
# 2SFCA
county_coords=np.column_stack([df.lon,df.lat])
hosp_coords=np.column_stack([hlon,hlat])
dist_mat=cdist(county_coords,hosp_coords)
CATCH=4.5
R_j=np.zeros(N_H)
for j in range(N_H):
    in_z=dist_mat[:,j]<=CATCH; denom=(df.population.values[in_z]/(dist_mat[in_z,j]+0.1)**2).sum()
    R_j[j]=hbeds[j]/denom if denom>0 else 0
A_i=np.zeros(N)
for i in range(N):
    in_z=dist_mat[i,:]<=CATCH
    if in_z.any():
        w=1/(dist_mat[i,in_z]+0.1)**2; A_i[i]=(R_j[in_z]*w).sum()
df["access_2sfca"]=A_i

# Compound disadvantage index (quintile-based scoring)
def quintile_score(v, higher_is_worse=True):
    q = pd.qcut(v, 5, labels=[1,2,3,4,5], duplicates="drop").astype(float)
    return q if higher_is_worse else 6-q

df["cdi"] = (quintile_score(df.cvd_rate) +
             quintile_score(df.pct_poverty) +
             quintile_score(df.pm25) +
             quintile_score(df.pct_uninsured) +
             quintile_score(df.access_2sfca, higher_is_worse=False)) / 5

df["cdi_cat"] = pd.cut(df.cdi, bins=[0,2,3,4,5.1], labels=["Low","Moderate","High","Very High"])
print("Compound Disadvantage Index:")
for cat in ["Low","Moderate","High","Very High"]:
    n_c=(df.cdi_cat==cat).sum(); pop_c=df.loc[df.cdi_cat==cat,"population"].sum()
    print(f"  {cat:12s}: {n_c:3d} counties | {pop_c:,} residents ({pop_c/df.population.sum()*100:.1f}%)")


## 7. Publication Summary Figure (8 panels, 300 dpi)

In [ ]:
fig=plt.figure(figsize=(22,16))
fig.suptitle("Spatial Health Inequity Atlas — CVD Mortality, Pollution & Access (N=200 Counties)",
             fontsize=15,y=1.01)
gs=gridspec.GridSpec(3,4,figure=fig,hspace=0.45,wspace=0.35)

CLUSTER_COLORS = {"HH":"#D65F5F","LL":"#4878CF","Other":"#B3CDE3","NS":"#E0E0E0"}
CDI_COLORS = {"Low":"#4878CF","Moderate":"#B3CDE3","High":"#FEC44F","Very High":"#D65F5F"}

# A — CVD Choropleth
ax_a=fig.add_subplot(gs[0,0])
sc=ax_a.scatter(df.lon,df.lat,c=df.cvd_rate,cmap="RdYlGn_r",s=80,alpha=0.85,edgecolors="none")
plt.colorbar(sc,ax=ax_a,label="Rate/100k"); ax_a.set_title("A  CVD Mortality Rate",fontweight="bold")

# B — EB-Smoothed SMR
ax_b=fig.add_subplot(gs[0,1])
sc2=ax_b.scatter(df.lon,df.lat,c=df.smr_eb,cmap="RdBu_r",vmin=0.6,vmax=1.6,s=80,alpha=0.85,edgecolors="none")
plt.colorbar(sc2,ax=ax_b,label="EB-SMR"); ax_b.set_title("B  EB-Smoothed SMR",fontweight="bold")

# C — LISA Cluster Map
ax_c=fig.add_subplot(gs[0,2])
colors_lisa=[CLUSTER_COLORS.get(c,"#E0E0E0") for c in df.lisa_cvd]
ax_c.scatter(df.lon,df.lat,c=colors_lisa,s=80,alpha=0.85,edgecolors="none")
for lab,col in CLUSTER_COLORS.items():
    n_l=(df.lisa_cvd==lab).sum()
    ax_c.plot([],[],"-",color=col,lw=6,label=f"{lab}(n={n_l})")
ax_c.legend(fontsize=6,loc="lower right"); ax_c.set_title("C  LISA Clusters",fontweight="bold")

# D — Scan Cluster
ax_d=fig.add_subplot(gs[0,3])
colors_scan=["#D65F5F" if v else "#E0E0E0" for v in df.in_scan_cluster]
ax_d.scatter(df.lon,df.lat,c=colors_scan,s=80,alpha=0.85,edgecolors="none")
cx=df.iloc[best_c].lon; cy=df.iloc[best_c].lat
circle=plt.Circle((cx,cy),best_r,fill=False,edgecolor="black",lw=2,ls="--")
ax_d.add_patch(circle); ax_d.set_title(f"D  Scan Statistic (LLR={best_llr:.1f},p={p_scan:.3f})",fontweight="bold")

# E — PM2.5 Surface
ax_e=fig.add_subplot(gs[1,0])
sc5=ax_e.scatter(df.lon,df.lat,c=df.pm25,cmap="YlOrBr",s=80,alpha=0.85,edgecolors="none")
plt.colorbar(sc5,ax=ax_e,label="µg/m³"); ax_e.set_title("E  PM2.5 Exposure",fontweight="bold")

# F — 2SFCA Access
ax_f=fig.add_subplot(gs[1,1])
sc6=ax_f.scatter(df.lon,df.lat,c=df.access_2sfca,cmap="RdYlGn",s=80,alpha=0.85,edgecolors="none")
ax_f.scatter(hlon,hlat,marker="P",c="black",s=40,zorder=5)
plt.colorbar(sc6,ax=ax_f,label="2SFCA score"); ax_f.set_title("F  2SFCA Healthcare Access",fontweight="bold")

# G — Compound Disadvantage
ax_g=fig.add_subplot(gs[1,2:])
colors_cdi=[CDI_COLORS[str(c)] for c in df.cdi_cat]
ax_g.scatter(df.lon,df.lat,c=colors_cdi,s=80,alpha=0.85,edgecolors="none")
ax_g.scatter(df.loc[cluster1|cluster2,"lon"],df.loc[cluster1|cluster2,"lat"],
             marker="*",c="gold",s=120,zorder=5,label="True CVD hotspot")
for lab,col in CDI_COLORS.items():
    ax_g.plot([],[],"-",color=col,lw=6,label=lab)
ax_g.legend(fontsize=7,loc="lower right")
ax_g.set_title("G  Compound Disadvantage Index (CDI)
(CVD + Poverty + PM2.5 + Uninsured + Poor Access)",fontweight="bold")

# H — Regression comparison
ax_h=fig.add_subplot(gs[2,:2])
methods_r=["OLS","Spatial Lag"]; coefs_r=[ols_pov,slm_pov]; true_r=1.20
bars_r=ax_h.bar(methods_r,coefs_r,color=["#4878CF","#D65F5F"],alpha=0.85,edgecolor="white",width=0.5)
ax_h.axhline(true_r,color="black",ls="--",lw=2,label=f"True β={true_r}")
ax_h.set_ylabel("Poverty coefficient (β)"); ax_h.set_title("H  Spatial Regression: Poverty->CVD",fontweight="bold")
for bar,val in zip(bars_r,coefs_r):
    ax_h.text(bar.get_x()+bar.get_width()/2,val+0.02,f"{val:.3f}",ha="center",fontsize=10)
ax_h.legend(fontsize=9)

# I — Moran scatter
ax_i=fig.add_subplot(gs[2,2:])
z_cvd=(df.cvd_rate-df.cvd_rate.mean())/df.cvd_rate.std()
Wz_cvd=W@z_cvd
ax_i.scatter(z_cvd,Wz_cvd,c=df.cvd_rate,cmap="RdYlGn_r",alpha=0.5,s=25,edgecolors="none")
m=np.polyfit(z_cvd,Wz_cvd,1)[0]
xl=np.linspace(z_cvd.min(),z_cvd.max(),100)
ax_i.plot(xl,m*xl,"k-",lw=2,label=f"Moran's I={I_cvd:.3f} (p={p_cvd:.3f})")
ax_i.axhline(0,color="gray",lw=0.8); ax_i.axvline(0,color="gray",lw=0.8)
ax_i.set_xlabel("z(CVD rate)"); ax_i.set_ylabel("W*z (spatial lag)")
ax_i.set_title(f"I  Moran Scatterplot (I={I_cvd:.3f}, p={p_cvd:.3f})",fontweight="bold")
ax_i.legend(fontsize=9)

for ax in [ax_a,ax_b,ax_c,ax_d,ax_e,ax_f,ax_g]:
    ax.set_xlabel("Lon",fontsize=7); ax.set_ylabel("Lat",fontsize=7)

plt.savefig("/tmp/mod07/capstone_spatial_atlas.png",bbox_inches="tight",dpi=300)
plt.show()
print("Saved: capstone_spatial_atlas.png (300 dpi)")


## 8. Methods Narrative Template

> **Spatial Epidemiology Methods**
>
> We conducted a county-level (N=200) spatial analysis of CVD mortality, environmental exposures, and healthcare access. Age-adjusted CVD mortality rates were empirical Bayes (EB) smoothed using a Gamma-Poisson conjugate model to stabilise estimates in low-population counties.
>
> Spatial autocorrelation was assessed using Global Moran's I with queen contiguity weights (W), evaluated against 499 permutation-based null replicates. Local spatial clusters were identified with LISA (p<0.05), and disease hot spots confirmed with the Kulldorff circular scan statistic (B=99 Monte Carlo permutations, 40% maximum population window).
>
> CVD rates were regressed on poverty, uninsured rate, and PM2.5 using OLS and a spatial lag model (SLM). PM2.5 exposure at the county level was estimated via inverse distance weighting (IDW, p=2) from point-monitor readings. Healthcare access was quantified using the Two-Step Floating Catchment Area (2SFCA) method with a 4.5° catchment and distance-decay power of 2.
>
> A Compound Disadvantage Index (CDI) was constructed by averaging quintile scores across CVD mortality, poverty rate, PM2.5, uninsured rate, and 2SFCA access, identifying counties bearing multiple simultaneous burdens. All analyses used Python 3.10 (scipy 1.11, statsmodels 0.14, sklearn 1.3).


## 9. Final Knowledge Check
1. The scan statistic detects a cluster with p=0.02 but the LISA map shows only 3 HH counties overlapping. How do you reconcile these findings?
2. SLM poverty coefficient is lower than OLS (0.95 vs 1.42). Explain conceptually why spatial adjustment reduces the estimate.
3. CDI identifies 28 counties with "Very High" compound disadvantage. A policy maker wants to intervene in all of them. What information would you add to prioritise among them?
4. The EB smoothing reduces SMR variance by 35%. What are the implications for mapping and for identifying outlier counties?
5. Design a study using ITS analysis (from NB-07 Module 06) to evaluate whether a telemedicine programme in HPSA counties reduced CVD mortality rates over 5 years.


In [ ]:
# Capstone summary statistics
print("CAPSTONE SUMMARY STATISTICS")
print("="*60)
print(f"Global Moran's I (CVD rate): {I_cvd:.4f} (p={p_cvd:.4f})")
print(f"LISA HH clusters: {sum(df.lisa_cvd=='HH')} | LL: {sum(df.lisa_cvd=='LL')} counties")
print(f"Scan cluster: LLR={best_llr:.2f} | p={p_scan:.3f} | n={len(best_in)} counties")
print(f"OLS poverty β={ols_pov:.3f} | SLM poverty β={slm_pov:.3f} | True β=1.20")
print(f"EB smoothing: SMR SD reduced {df.smr.std():.4f} -> {df.smr_eb.std():.4f}")
print(f"  Variance reduction: {(1-df.smr_eb.std()/df.smr.std())*100:.1f}%")
print(f"Compound Disadvantage Index (Very High): {(df.cdi_cat=='Very High').sum()} counties")
vhigh_pop = df.loc[df.cdi_cat=='Very High','population'].sum()
print(f"  Population in Very High CDI: {vhigh_pop:,} ({vhigh_pop/df.population.sum()*100:.1f}%)")
print("="*60)


---
## Module 07 Complete

**NB-01** Spatial data & choropleth: data types (point/polygon/raster), GeoDataFrame, quintile/Jenks classification, age-standardised rates, SMR computation, bivariate choropleth, hospital spatial join  
**NB-02** Spatial autocorrelation: queen/rook/distance weights matrices, Global Moran's I with permutation inference, Moran scatterplot, LISA (HH/LL/HL/LH), Getis-Ord G* hot spot analysis, Moran's I on OLS residuals  
**NB-03** Spatial regression & GWR: OLS residual diagnostics, LM lag/error tests, spatial lag model (SLM, ML estimation), spatial error model (SEM), GWR from scratch with bandwidth sensitivity, spatially varying coefficient maps  
**NB-04** Disease mapping & Bayesian smoothing: small-area estimation problem, Gamma-Poisson empirical Bayes, BYM/ICAR model structure, Moran eigenvector spatial smoothing, KDE risk surface, PyMC BYM starter code  
**NB-05** Cluster detection: Kulldorff circular scan statistic, Poisson LLR, Monte Carlo permutation inference, cluster performance evaluation (sensitivity/specificity/PPV), LISA vs scan comparison, secondary cluster detection  
**NB-06** Environmental epidemiology: IDW/RBF exposure interpolation, ecological regression, ecological fallacy, environmental justice (quintile and concentration index), individual vs aggregate effect comparison  
**NB-07** Health service access: PPR, nearest-distance, 2SFCA method, HPSA-style shortage area delineation, access-deprivation correlation, concentration index, Gini coefficient for access distribution  
**NB-08** Capstone: full pipeline → EB smoothing → Moran's I → LISA → scan stat → SLM → 2SFCA → CDI → 8-panel 300 dpi atlas → methods narrative

**Next → Module 08: Reproducible Research & Deployment**
